# PakRetail DW — Retail Sales & COD Delivery Intelligence
**Semester 7 | Data Warehousing and Big Data Analytics**

A fictional Pakistani omnichannel retailer needs a unified analytical store to compare sales across Lahore, Karachi, Islamabad, Faisalabad, Multan and Peshawar, monitor cash-on-delivery (COD) failures and prioritize orders for confirmation. All records are synthetic, reproducible and denominated in PKR; no actual company or customer data is used. Run cells in order in Google Colab (CPU runtime).

## 1. Architecture and course concepts
Raw CSV + JSON → staging (pandas inspection, Spark cleaning) → dimensional warehouse (Parquet) → Spark SQL OLAP → reporting marts → MLlib risk model.

The fact grain is one order containing one product; dimensions are date, customer, product and branch. Foreign keys connect facts to dimensions. Customer dimension is a current snapshot (Type 1); this is a full-refresh batch prototype, not an incremental production warehouse. The five Vs are represented by configurable volume, batch arrival (velocity), CSV/JSON/text variety, data validation (veracity) and business KPIs (value). Colab runs local Spark threads: driver builds DAGs, executors process partitions, actions trigger lazy transformations and groupBy/join can cause shuffles. HDFS/YARN are production alternatives, not deployed here.

In [ ]:
# Import setup utilities and install Spark only if it is missing.
import importlib.util, subprocess, sys, os
if importlib.util.find_spec('pyspark') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark==4.0.1'])
# Import the libraries for file handling, data analysis and visualization.
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
# Import Spark SQL and machine learning components.
from pyspark.sql import SparkSession, functions as F, Window, types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
# Start a local Spark session with two worker threads and four shuffle partitions.
spark = (SparkSession.builder.master('local[2]').appName('PakRetailDW')
         .config('spark.sql.shuffle.partitions','4').config('spark.driver.memory','2g').getOrCreate())
# Reduce log noise and create output folders for raw data, warehouse and reports.
spark.sparkContext.setLogLevel('ERROR')
BASE = Path('pakretail_output').resolve(); RAW = BASE/'raw'; WH = BASE/'warehouse'; MART = BASE/'marts'
for p in (RAW, WH, MART): p.mkdir(parents=True, exist_ok=True)
print(spark.version, BASE)

## 2. Generate local-context source systems
20,000 fictional orders in 2025, 1,000 customers, 60 products and six branches. Failure probability is deliberately simulated using information known at checkout plus random noise. This is not a claim about actual Pakistani failure rates. Missing discounts, inconsistent payment spelling, duplicates and invalid quantities are inserted to demonstrate ETL.

In [ ]:
# Set a fixed random seed so the synthetic dataset can be reproduced.
rng = np.random.default_rng(42)
N = 20000
# Define the Pakistani branch locations and generate customer and product records.
cities = ['Lahore','Karachi','Islamabad','Faisalabad','Multan','Peshawar']
branches = pd.DataFrame({'branch_id':range(1,7),'city':cities,'province':['Punjab','Sindh','ICT','Punjab','Punjab','Khyber Pakhtunkhwa']})
customers = pd.DataFrame({'customer_id':range(1,1001),'segment':rng.choice(['New','Regular','Loyal'],1000,p=[.3,.5,.2])})
products = pd.DataFrame({'product_id':range(1,61),'product_name':[f'Item {i:03}' for i in range(1,61)],'category':rng.choice(['Fashion','Electronics','Grocery','Home','Beauty'],60),'unit_price':rng.integers(400,15001,60)})
# Generate one row per order with checkout-time attributes.
orders = pd.DataFrame({'order_id':range(1,N+1),'order_date':pd.Timestamp('2025-01-01')+pd.to_timedelta(rng.integers(0,365,N),unit='D'),'customer_id':rng.integers(1,1001,N),'product_id':rng.integers(1,61,N),'branch_id':rng.integers(1,7,N),'quantity':rng.integers(1,5,N),'payment':rng.choice(['COD','Card','Wallet'],N,p=[.65,.2,.15]),'distance_km':rng.integers(1,61,N),'discount_pct':rng.choice([0,5,10,15],N,p=[.6,.2,.15,.05])})
# Join product prices and customer segments to simulate delivery outcomes.
orders = orders.merge(products[['product_id','unit_price']],on='product_id',validate='many_to_one').merge(customers,on='customer_id',validate='many_to_one')
# Convert simulated failure probabilities into binary labels and complaint text.
logit = -3.4 + 1.15*(orders.payment=='COD') + .035*orders.distance_km + .65*(orders.segment=='New') + .000025*orders.unit_price*orders.quantity
orders['failed'] = (rng.random(N)<1/(1+np.exp(-logit))).astype(int)
orders['complaint'] = np.where(orders.failed==1,rng.choice(['delivery refused','address not found','customer unavailable'],N),'')
# Introduce deliberate data quality issues for the ETL demonstration.
orders = orders.drop(columns='segment')
orders.loc[:49,'discount_pct']=np.nan
orders.loc[50:99,'payment']=' cod '
orders.loc[100:109,'quantity']=0
# Export source tables to CSV and JSON and inspect the raw data.
orders = pd.concat([orders,orders.iloc[:100]],ignore_index=True)
orders.to_csv(RAW/'orders.csv',index=False); customers.to_csv(RAW/'customers.csv',index=False); products.to_csv(RAW/'products.csv',index=False); branches.to_json(RAW/'branches.json',orient='records',lines=True)
print(orders.shape); print(orders.isna().sum()); print('Exact duplicates:',orders.duplicated().sum()); display(orders.head())

## 3. Extract, transform and validate
Missing discount means no discount in this simulation. Invalid quantities are quarantined, not silently repaired. Standardize payment labels, cast dates, remove exact duplicates and check primary/foreign keys. Monetary values use decimal arithmetic to avoid floating-point rounding issues.

In [ ]:
# Extract the raw CSV into a Spark DataFrame and inspect its schema.
raw = spark.read.csv(str(RAW/'orders.csv'),header=True,inferSchema=True)
raw.printSchema(); raw.show(5)
print('Raw rows:',raw.count())
# Remove exact duplicate rows and verify unique order identifiers.
unique = raw.dropDuplicates()
assert unique.count()==N
assert unique.select('order_id').distinct().count()==N
# Standardize dates and payment labels; replace missing discounts with zero.
stage = (unique.withColumn('order_date',F.to_date('order_date')).withColumn('payment',F.upper(F.trim('payment'))).withColumn('discount_pct',F.coalesce(F.col('discount_pct'),F.lit(0))))
# Build a validation rule for quantities, prices, payment, discounts and labels.
valid = (F.col('quantity')>0)&(F.col('unit_price')>0)&F.col('payment').isin('COD','CARD','WALLET')&F.col('discount_pct').between(0,100)&F.col('failed').isin(0,1)&F.col('order_date').isNotNull()
# Treat missing validation results as invalid and quarantine rejected records.
valid = F.coalesce(valid,F.lit(False))
rejects = stage.filter(~valid); rejects.write.mode('overwrite').parquet(str(BASE/'rejected'))
# Cache accepted records and reconcile accepted and rejected row counts.
clean = stage.filter(valid).cache()
assert clean.count()+rejects.count()==N
print('Accepted:',clean.count(),'Rejected:',rejects.count())
clean.select('quantity','unit_price','distance_km').describe().show()

## 4. Build the star schema
`fact_orders(order_id, date_key, customer_id, product_id, branch_id, ...)` joins to `dim_date`, `dim_customer`, `dim_product`, `dim_branch`. Gross value is quantity × checkout price; net order value subtracts discount. Recognized revenue is net value only for successful deliveries; this is a simplified operational revenue proxy, not audited accounting. Failed orders are retained for analytics.

In [ ]:
# Load the customer, product and branch dimensions from the source files.
dim_customer=spark.read.csv(str(RAW/'customers.csv'),header=True,inferSchema=True)
dim_product=spark.read.csv(str(RAW/'products.csv'),header=True,inferSchema=True).drop('unit_price')
dim_branch=spark.read.json(str(RAW/'branches.json'))
# Create a date dimension containing every day of 2025 and its calendar attributes.
dim_date=(spark.range(365).select(F.date_add(F.lit('2025-01-01').cast('date'),F.col('id').cast('int')).alias('date')).withColumn('date_key',F.date_format('date','yyyyMMdd').cast('int')).withColumn('year',F.year('date')).withColumn('month',F.month('date')).withColumn('quarter',F.quarter('date')))
# Build the fact table and calculate gross value, discounted value and recognized revenue.
fact=(clean.withColumn('date_key',F.date_format('order_date','yyyyMMdd').cast('int')).withColumn('gross_pkr',(F.col('quantity')*F.col('unit_price')).cast('decimal(16,2)')).withColumn('net_pkr',F.round(F.col('gross_pkr')*(100-F.col('discount_pct'))/100,2).cast('decimal(16,2)')).withColumn('revenue_pkr',F.when(F.col('failed')==0,F.col('net_pkr')).otherwise(F.lit(0)).cast('decimal(16,2)')).drop('order_date','complaint'))
# Register all five star-schema tables and check key uniqueness and referential integrity.
tables={'fact_orders':fact,'dim_customer':dim_customer,'dim_product':dim_product,'dim_branch':dim_branch,'dim_date':dim_date}
for name, key in [('dim_customer','customer_id'),('dim_product','product_id'),('dim_branch','branch_id'),('dim_date','date_key')]:
    d=tables[name]; assert d.count()==d.select(key).distinct().count(); assert fact.join(d,key,'left_anti').count()==0
# Write each table to columnar Parquet storage and register temporary SQL views.
for name, df in tables.items():
    df.write.mode('overwrite').parquet(str(WH/name)); spark.read.parquet(str(WH/name)).createOrReplaceTempView(name)
# Verify fact-table grain and revenue consistency.
assert fact.select('order_id').distinct().count()==clean.count()
assert fact.filter(F.col('revenue_pkr')>F.col('net_pkr')).count()==0
print('Warehouse persisted and validated.')

## 5. OLAP queries and analytical marts
Roll up monthly revenue, slice COD orders, dice by city/category, rank categories within cities and drill down from quarter to month. SQL works on persisted warehouse views rather than raw files.

In [ ]:
# Define SQL queries for city, monthly and payment KPIs, category ranks, roll-up and COD analysis.
queries={
'city_kpis': """SELECT b.city, COUNT(*) orders, SUM(f.revenue_pkr) revenue_pkr, ROUND(100*AVG(f.failed),2) failure_pct FROM fact_orders f JOIN dim_branch b USING(branch_id) GROUP BY b.city ORDER BY revenue_pkr DESC""",
'monthly_kpis': """SELECT d.year,d.month,SUM(f.revenue_pkr) revenue_pkr,COUNT(*) orders,ROUND(100*AVG(f.failed),2) failure_pct FROM fact_orders f JOIN dim_date d USING(date_key) GROUP BY d.year,d.month ORDER BY d.year,d.month""",
'payment_kpis': """SELECT payment,COUNT(*) orders,SUM(failed) failed_orders,ROUND(100*AVG(failed),2) failure_pct,SUM(CASE WHEN failed=1 THEN net_pkr ELSE 0 END) failed_order_value_pkr FROM fact_orders GROUP BY payment""",
'category_rank': """WITH s AS (SELECT b.city,p.category,SUM(f.revenue_pkr) revenue_pkr FROM fact_orders f JOIN dim_branch b USING(branch_id) JOIN dim_product p USING(product_id) GROUP BY b.city,p.category) SELECT *,DENSE_RANK() OVER(PARTITION BY city ORDER BY revenue_pkr DESC) category_rank FROM s ORDER BY city,category_rank""",
'quarter_rollup': """SELECT d.year,d.quarter,d.month,SUM(f.revenue_pkr) revenue_pkr FROM fact_orders f JOIN dim_date d USING(date_key) GROUP BY ROLLUP(d.year,d.quarter,d.month) ORDER BY d.year,d.quarter,d.month""",
'cod_city_category': """SELECT b.city,p.category,COUNT(*) orders,ROUND(100*AVG(f.failed),2) failure_pct FROM fact_orders f JOIN dim_branch b USING(branch_id) JOIN dim_product p USING(product_id) WHERE payment='COD' GROUP BY b.city,p.category ORDER BY failure_pct DESC"""}
# Execute each query and export the small aggregated result as a CSV reporting mart.
for name, query in queries.items():
    print(name); result=spark.sql(query); result.show(12); result.toPandas().to_csv(MART/f'{name}.csv',index=False)
# Inspect the Spark physical execution plan for the city KPI query.
print('Execution plan:'); spark.sql(queries['city_kpis']).explain()

In [ ]:
# Read the aggregated reporting marts into pandas for plotting.
monthly=pd.read_csv(MART/'monthly_kpis.csv'); city=pd.read_csv(MART/'city_kpis.csv'); pay=pd.read_csv(MART/'payment_kpis.csv')
# Create and save charts for monthly revenue, city revenue and payment failure rates.
for data,x,y,title,kind in [(monthly,'month','revenue_pkr','Monthly recognized revenue (PKR)','line'),(city,'city','revenue_pkr','Revenue by city (PKR)','bar'),(pay,'payment','failure_pct','Delivery failure by payment (%)','bar')]:
    ax=data.plot(x=x,y=y,kind=kind,legend=False,title=title,figsize=(8,4)); ax.set_ylabel(y); plt.tight_layout(); plt.savefig(MART/f'{y}_{x}.png',dpi=140); plt.show()

## 6. RDD, MapReduce and partitions
RDD flatMap tokenizes complaint text, map creates pairs and reduceByKey combines counts across partitions. Collect only a small aggregated result. Complaints are available after delivery and must never enter the predictive model.

In [ ]:
# Filter failed deliveries and split complaint text into individual words.
complaints=clean.filter(F.col('failed')==1).select('complaint').rdd.flatMap(lambda row: row[0].split())
# Map each word to a count of one and aggregate counts across partitions.
counts=complaints.map(lambda word:(word,1)).reduceByKey(lambda a,b:a+b)
# Display frequent words and compare partition counts before and after repartitioning.
print('RDD partitions:',counts.getNumPartitions()); print(counts.takeOrdered(10,key=lambda x:-x[1]))
print('Fact partitions:',fact.rdd.getNumPartitions()); print('After repartition:',fact.repartition(4,'branch_id').rdd.getNumPartitions())

## 7. MLlib extension: predict failed deliveries
The label is failed (1) versus successful (0). Use only checkout-time features: payment, customer segment, city, product category, quantity, unit price, discount and distance. Split by date: January–September train, October validation, November–December test. Fit encoders/scalers on train only. Select model by validation PR-AUC, then evaluate the selected model once on test. No post-delivery complaint or revenue is used. Synthetic labels mean scores cannot establish real-world effectiveness.

In [ ]:
# Join warehouse dimensions to retrieve model features and create the binary target label.
data=spark.sql("""SELECT f.*,d.date,c.segment,b.city,p.category FROM fact_orders f JOIN dim_date d USING(date_key) JOIN dim_customer c USING(customer_id) JOIN dim_branch b USING(branch_id) JOIN dim_product p USING(product_id)""").withColumn('label',F.col('failed').cast('double')).withColumn('discount_pct',F.col('discount_pct').cast('double')).cache()
# Split records chronologically into training, validation and test sets to avoid future leakage.
train=data.filter(F.col('date')<'2025-10-01'); val=data.filter((F.col('date')>='2025-10-01')&(F.col('date')<'2025-11-01')); test=data.filter(F.col('date')>='2025-11-01')
# Inspect the class distribution of each split.
for name,split in [('train',train),('validation',val),('test',test)]:
    print(name); split.groupBy('label').count().show()
# Define checkout-time categorical and numeric features; exclude post-delivery information.
cat=['payment','segment','city','category']; num=['quantity','unit_price','discount_pct','distance_km']
# Encode categories, assemble feature vectors and scale numeric inputs inside the pipeline.
indices=[StringIndexer(inputCol=c,outputCol=c+'_idx',handleInvalid='keep') for c in cat]
encoder=OneHotEncoder(inputCols=[c+'_idx' for c in cat],outputCols=[c+'_vec' for c in cat],handleInvalid='keep')
assembler=VectorAssembler(inputCols=num+[c+'_vec' for c in cat],outputCol='raw_features')
scaler=StandardScaler(inputCol='raw_features',outputCol='features',withMean=False)
# Define two classifiers and evaluation metrics for comparing them.
models={'logistic':LogisticRegression(maxIter=25,regParam=.05),'random_forest':RandomForestClassifier(numTrees=25,maxDepth=5,seed=42)}
pr=BinaryClassificationEvaluator(metricName='areaUnderPR'); roc=BinaryClassificationEvaluator(metricName='areaUnderROC')
fitted={}; scores={}
# Fit each pipeline on training data only and select using validation PR-AUC.
for name,est in models.items():
    fitted[name]=Pipeline(stages=indices+[encoder,assembler,scaler,est]).fit(train)
    scores[name]=pr.evaluate(fitted[name].transform(val))
# Evaluate the selected model on the held-out test set.
print('Validation PR-AUC:',scores); best=max(scores,key=scores.get); print('Selected:',best)
pred=fitted[best].transform(test).cache()
metrics={'model':best,'pr_auc':pr.evaluate(pred),'roc_auc':roc.evaluate(pred)}
for m in ['accuracy','f1','weightedPrecision','weightedRecall']:
    metrics[m]=MulticlassClassificationEvaluator(metricName=m).evaluate(pred)
# Compute confusion matrix and failure-class precision and recall.
cm=pred.groupBy('label','prediction').count().toPandas(); display(cm)
lookup={(int(r.label),int(r.prediction)):int(r['count']) for _,r in cm.iterrows()}
tp=lookup.get((1,1),0); fp=lookup.get((0,1),0); fn=lookup.get((1,0),0)
# Compare against the majority-class baseline, export metrics and save the trained pipeline.
metrics['failure_precision']=tp/max(1,tp+fp); metrics['failure_recall']=tp/max(1,tp+fn)
metrics['majority_baseline_accuracy']=test.filter(F.col('label')==0).count()/test.count()
print(metrics); pd.DataFrame([metrics]).to_csv(MART/'model_metrics.csv',index=False); cm.to_csv(MART/'confusion_matrix.csv',index=False)
fitted[best].write().overwrite().save(str(BASE/'delivery_risk_model'))

## 8. Actionable outputs and validation
Risk scores support voluntary order confirmation rather than automatic rejection. The score is a model probability estimate, not a guaranteed outcome. Compare failure rates and failed order value by payment and city before deciding where to pilot interventions. A real pilot should measure confirmation cost, incremental successful deliveries and customer experience.

In [ ]:
# Extract the predicted failure probability and rank test orders for voluntary review.
from pyspark.ml.functions import vector_to_array
risk=pred.select('order_id','payment','city',vector_to_array('probability')[1].alias('failure_probability')).orderBy(F.desc('failure_probability'))
# Export the top 100 candidates and display a small sample.
risk.limit(100).toPandas().to_csv(MART/'risk_review_queue.csv',index=False)
print('Top risk review candidates:'); risk.show(10)
# Check that the fact count and recognized revenue satisfy warehouse rules.
assert spark.sql('SELECT COUNT(*) FROM fact_orders').first()[0]==N-10
assert spark.sql('SELECT SUM(revenue_pkr) FROM fact_orders').first()[0]==spark.sql('SELECT SUM(revenue_pkr) FROM fact_orders WHERE failed=0').first()[0]
print('All warehouse checks passed. Outputs:',BASE)
# Optional: download the generated warehouse and reports in Colab.
# Package all generated output files into a ZIP archive.
import shutil
archive=shutil.make_archive('pakretail_results','zip',BASE)
print(archive)

## 9. Conclusions and viva notes
- A warehouse integrates source systems at a stable grain; a data lake stores raw heterogeneous inputs. Parquet is columnar storage, not a managed warehouse service.
- Spark local mode demonstrates partitioned execution but does not benchmark a multi-node cluster. A production design can use HDFS or object storage, a cluster manager and scheduled incremental ingestion.
- The warehouse supports descriptive, diagnostic and predictive analytics. The ML model is an optional analytical layer, not a replacement for ETL and OLAP.
- The synthetic data limits external validity. Real courier records, consent/access controls, drift monitoring, time-based validation and cost-benefit testing are required before operational deployment.

**Course references:** All 12 supplied lecture files are mapped in the repository README. The star schema and OLAP design are additional project-specific warehousing concepts.